# Checkpoint sampling runtime

Measure sampling for one saved checkpoint. Project the same workload across all saved checkpoints.

This notebook does not compute metrics. It does not save the generated sample bank.

In [ ]:
# Import the benchmark and display tools.
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "mem_param_src").is_dir()
)

from mem_param_src.runners.artifacts import list_checkpoints, read_json
from mem_param_src.runners.sampling_benchmark import benchmark_checkpoint_sampling

## Settings

Select one preset. Start with 256 samples. Use 10,000 for the exact paper-scale checkpoint time.

In [ ]:
# Select the saved run and timing size.
RUN_PRESETS = {
    "celeba32": PROJECT_ROOT / "outputs/job_sets/celeba32_unet_3x3_1m/20260927T191224Z_2530324/runs/18_n2048_x_x_s0",
    "mnist32": PROJECT_ROOT / "outputs/job_sets/mnist_diagonal_1m/20260927T165106Z_1991997/runs/0_mnist32_unet_s_n512_x_x_s0",
}

RUN_KEY = "celeba32"  # Choose celeba32 or mnist32.
RUN_DIR = RUN_PRESETS[RUN_KEY]
CHECKPOINT_STEP = 1_000_000
MEASURE_COUNT = 256  # Set 10_000 for an exact checkpoint time.
TARGET_COUNT = 10_000  # Keep the paper-scale projection fixed.
BATCH_SIZE = None  # None uses the saved evaluation batch size.
WARMUP_BATCHES = 1  # Exclude one warm-up batch from timing.
PREVIEW_COUNT = 16
DEVICE = "cuda"
OUTPUT_DIR = None  # Set a new directory to save timing tables.

In [ ]:
# Inspect the run before sampling.
config = read_json(RUN_DIR / "config.json")
run = read_json(RUN_DIR / "run.json")
checkpoints = list_checkpoints(RUN_DIR)

run_table = pd.DataFrame([
    {
        "dataset": config["data"]["dataset"],
        "n_train": config["data"]["n_train"],
        "seed": config["training"]["seed"],
        "prediction/loss": f"{config['framework']['prediction_space']}/{config['framework']['loss_space']}",
        "checkpoint": CHECKPOINT_STEP,
        "saved_checkpoints": len(checkpoints),
        "run_state": run["state"],
    }
])
display(run_table)

In [ ]:
# Measure complete sampling batches for one checkpoint.
result = benchmark_checkpoint_sampling(
    RUN_DIR,
    checkpoint_step=CHECKPOINT_STEP,
    measure_count=MEASURE_COUNT,
    target_count=TARGET_COUNT,
    batch_size=BATCH_SIZE,
    warmup_batches=WARMUP_BATCHES,
    device=DEVICE,
    output_dir=OUTPUT_DIR,
    preview_count=PREVIEW_COUNT,
)

In [ ]:
# Show the measured throughput and projected sampling times.
summary = result["summary"]
timing = summary["timing"]
report = pd.DataFrame([
    {
        "measured samples": timing["measured_samples"],
        "sampling seconds": timing["sampling_seconds"],
        "samples/second": timing["samples_per_second"],
        "10k checkpoint minutes": timing["one_checkpoint_sampling_minutes"],
        "checkpoint timing": timing["one_checkpoint_timing_kind"],
        "saved checkpoints": timing["checkpoint_count"],
        "all checkpoint hours": timing["all_checkpoints_sampling_hours"],
        "peak GPU GiB": summary["peak_cuda_memory_gib"],
    }
])
display(report.round(3))
display(pd.DataFrame([summary["diagnostics"]]).round(5))

In [ ]:
# Display a small preview outside the timed section.
images = result["preview"].mul(0.5).add(0.5).clamp(0.0, 1.0)
columns = 4
rows = max(1, (len(images) + columns - 1) // columns)
figure, axes = plt.subplots(rows, columns, figsize=(8, 2 * rows))
axes = axes.reshape(-1)
for index, axis in enumerate(axes):
    axis.axis("off")
    if index >= len(images):
        continue
    image = images[index]
    if image.shape[0] == 1:
        axis.imshow(image[0], cmap="gray", vmin=0.0, vmax=1.0)
    else:
        axis.imshow(image.permute(1, 2, 0))
figure.suptitle(f"{RUN_KEY}: step {CHECKPOINT_STEP:,}")
figure.tight_layout()